# Stochastic Calculus from Zero · Episode 21: Girsanov's theorem

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The share** (Series 3's): priced at 100, 20% volatility, a 3.60% risk-free rate (continuous), a call with strike 105 expiring in a year, and an illustrative real-world drift of 8%. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The theorem · 4. The market price of risk · 5. Checks on simulated paths · 6. The price · 7. Back to the coin world · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 21,
    "quantlib_version": ql.__version__,
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. Last episode's answer

Euler's strong error falls like $\sqrt{\Delta t}$: halving it takes $4\times$ the steps, $4 \times 1{,}024 = 4{,}096$.

In [ ]:
out["prev"] = {"answer": 4096}

## 3. The theorem

Let $W$ be a Brownian motion under the real-world measure $P$, and $\theta$ a constant (more generally, an adapted process with $E\big[e^{\frac12\int_0^T\theta_t^2\,dt}\big] < \infty$, **Novikov's condition**). Define new weights on the paths by

$$\frac{dQ}{dP} = Z_T = \exp\!\big(-\theta W_T - \tfrac12\theta^2 T\big).$$

**Girsanov's theorem** (1960): under $Q$, the process $\widetilde W_t = W_t + \theta t$ is a Brownian motion.

The paths don't change; their weights do. A path that ends high ($W_T$ large) gets a small weight when $\theta > 0$, so under $Q$ the paths drift down by $\theta$ per unit time relative to $P$. Proof for constant $\theta$, via moment generating functions: in the book.

## 4. The market price of risk

Our share: $dS = \mu S\,dt + \sigma S\,dW$ under $P$. Choose

$$\theta = \frac{\mu - r}{\sigma},$$

the **market price of risk** (extra drift per unit of volatility). Then $dW = d\widetilde W - \theta\,dt$, and

$$dS = \mu S\,dt + \sigma S\,(d\widetilde W - \theta\,dt) = rS\,dt + \sigma S\,d\widetilde W .$$

Under $Q$ the share grows at the risk-free rate: $Q$ is the **risk-neutral measure**, and $e^{-rt}S_t$ is a $Q$-martingale.

In [ ]:
S0, VOL, R, MU, K, T = 100.0, 0.20, 0.036, 0.08, 105.0, 1.0
theta = (MU - R) / VOL
out["inputs"] = {"mu_pct": MU * 100, "r_pct": R * 100, "vol_pct": VOL * 100, "K": K, "theta": theta, "year": 1960}
theta

## 5. Checks on simulated paths

400,000 real-world outcomes $W_T$, each with its weight $Z_T$. Under $Q$, averages are weighted averages: $E_Q[X] = E_P[Z_T X]$.

In [ ]:
rng = np.random.default_rng(seed)
n = 400_000
WT = rng.standard_normal(n) * math.sqrt(T)
Z = np.exp(-theta * WT - 0.5 * theta ** 2 * T)
ST = S0 * np.exp((MU - 0.5 * VOL ** 2) * T + VOL * WT)          # real-world outcomes
Wq = WT + theta * T
checks = {"paths": n, "E_Z": float(Z.mean()), "E_Z_se": float(Z.std() / math.sqrt(n)),
          "E_ZW": float((Z * WT).mean()), "E_ZW_theory": -theta * T,
          "Q_mean_Wq": float((Z * Wq).mean()), "Q_var_Wq": float((Z * Wq ** 2).mean() - (Z * Wq).mean() ** 2),
          "P_mean_S": float(ST.mean()), "Q_mean_S": float((Z * ST).mean()), "forward": S0 * math.exp(R * T)}
assert abs(checks["E_Z"] - 1) < 4 * checks["E_Z_se"]
assert abs(checks["E_ZW"] + theta) < 0.01 and abs(checks["Q_mean_Wq"]) < 0.01 and abs(checks["Q_var_Wq"] - 1) < 0.01
assert abs(checks["Q_mean_S"] / checks["forward"] - 1) < 0.003
out["checks"] = checks
pd.Series(checks)

The pictures: where the share ends under $P$ (plain histogram) and under $Q$ (each outcome counted with weight $Z_T$), against the two lognormal densities.

In [ ]:
bins = np.arange(40, 200.01, 4)
mid = (bins[:-1] + bins[1:]) / 2
hp, _ = np.histogram(ST, bins=bins)
hq, _ = np.histogram(ST, bins=bins, weights=Z)
def lognormal(s, drift):
    m, v = math.log(S0) + (drift - 0.5 * VOL ** 2) * T, VOL ** 2 * T
    return np.exp(-(np.log(s) - m) ** 2 / (2 * v)) / (s * math.sqrt(2 * math.pi * v))
xs = np.linspace(40, 200, 161)
out["hist"] = {"x": mid.tolist(), "p": (hp / n).round(5).tolist(), "q": (hq / n).round(5).tolist(), "width": 4,
               "curve_x": xs.tolist(), "curve_p": (lognormal(xs, MU) * 4).round(5).tolist(),
               "curve_q": (lognormal(xs, R) * 4).round(5).tolist(),
               "ticks": [{"x": x, "label": str(x)} for x in range(40, 201, 40)]}
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(mid, hp / n, width=4, alpha=0.5, label="P"); ax.bar(mid, hq / n, width=4, alpha=0.5, label="Q (weighted)")
ax.axvline(K, color="k", ls="--"); ax.legend(); plt.show()

## 6. The price

The call pays $\max(S_T - K, 0)$. Discounting the **real-world** average payoff gives the wrong price; weighting by $Z_T$ gives the Black-Scholes price.

In [ ]:
pay = math.exp(-R * T) * np.maximum(ST - K, 0)
today = ql.Date(29, 9, 2026); ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
proc = ql.BlackScholesMertonProcess(ql.QuoteHandle(ql.SimpleQuote(S0)), flat(0.0), flat(R),
                                    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), VOL, dc)))
opt = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(today + 365))
opt.setPricingEngine(ql.AnalyticEuropeanEngine(proc))
price = {"naive": float(pay.mean()), "naive_se": float(pay.std() / math.sqrt(n)),
         "girsanov": float((Z * pay).mean()), "girsanov_se": float((Z * pay).std() / math.sqrt(n)),
         "bs": float(bs_call(S0, K, T, R, VOL)[0]), "ql": opt.NPV()}
assert abs(price["bs"] - price["ql"]) < 1e-6
assert abs(price["girsanov"] - price["bs"]) < 3 * price["girsanov_se"]
out["price"] = price
pd.Series(price)

## 7. Back to the coin world

Part 1, Episode 8 changed the odds on the coin world's eight paths: each path's weight under $Q$ was its weight under $P$ times $Z = \prod (q/p \text{ or } (1-q)/(1-p))$. Girsanov's $Z_T$ is the same idea with infinitely many tiny tosses: a heads-heavy (high) path is down-weighted, a tails-heavy path up-weighted, by exactly enough to move the drift from $\mu$ to $r$. And in both, the up and down move sizes (the volatility) stay the same.

## 8. This episode's question

If $\mu = r$, then $\theta = 0$ and $Z_T = 1$: the real world is already risk-neutral.

In [ ]:
out["question"] = {"theta": 0.0, "Z": 1.0}

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Show $E_P[Z_T] = 1$, using $E[e^{cW_T}] = e^{\frac12c^2T}$.

<details><summary>Answer</summary>

$E[e^{-\theta W_T}]e^{-\frac12\theta^2T} = e^{\frac12\theta^2T}e^{-\frac12\theta^2T} = 1$.

</details>

**Exercise 2 (By hand).** Show $E_Q[W_T] = -\theta T$, so $E_Q[W_T + \theta T] = 0$. (Hint: $E_P[W e^{cW}]$ is the derivative of the MGF in $c$.)

<details><summary>Answer</summary>

$E_P[W_Te^{cW_T}] = \frac{d}{dc}e^{\frac12c^2T} = cT e^{\frac12c^2T}$. With $c = -\theta$ and the factor $e^{-\frac12\theta^2T}$: $E_Q[W_T] = -\theta T$.

</details>

**Exercise 3 (By hand).** Why can't a change of measure change the volatility?

<details><summary>Answer</summary>

Volatility is read off the quadratic variation, $\sum(\Delta S)^2 \to \int\sigma^2S^2\,dt$, which is a pathwise limit. $Q$ and $P$ agree on which events have probability 1 (they are equivalent), so the limit is the same under both.

</details>

**Exercise 4 (In code).** Using the weights, check that under $Q$ the log return $\ln(S_T/S_0)$ has mean $(r - \tfrac12\sigma^2)T$ and variance $\sigma^2T$.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
L = np.log(ST / S0)
qm = float((Z * L).mean()); qv = float((Z * L ** 2).mean() - qm ** 2)
assert abs(qm - (R - 0.5 * VOL ** 2) * T) < 0.002 and abs(qv / (VOL ** 2 * T) - 1) < 0.02
print(f"Q mean {qm:.4f} (theory {(R - 0.5 * VOL ** 2) * T:.4f}), Q variance {qv:.4f} (theory {VOL ** 2 * T:.4f})")

**Exercise 5 (In code).** Redo the price with a real-world drift of 15%. What is $\theta$, and does the Z-weighted price change?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
mu2 = 0.15
th2 = (mu2 - R) / VOL
W2 = np.random.default_rng(5).standard_normal(n)
S2 = S0 * np.exp((mu2 - 0.5 * VOL ** 2) * T + VOL * W2)
Z2 = np.exp(-th2 * W2 - 0.5 * th2 ** 2 * T)
p2 = Z2 * math.exp(-R * T) * np.maximum(S2 - K, 0)
se2 = p2.std() / math.sqrt(n)
assert abs(p2.mean() - price["bs"]) < 3 * se2
print(f"theta {th2:.2f}; weighted price {p2.mean():.3f} ± {se2:.3f}, Black-Scholes {price['bs']:.3f}: the drift drops out")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")